In [ ]:
**Alunos:** Eduardo Barbosa
            Lucas Antonio Cunha Rodrigues da Silva
            Luiz Girotto  
            Marcos Vinícius Beregula

**Curso:** Ciência de Dados e IA – 3º ano  
**Disciplina:** Aprendizado Supervisionado  
**Professor:** Bruno Faiçal  
**Instituição:** Universidade Estadual de Londrina (UEL)  
**Data:** 05/06/2026  

/opt/anaconda3/envs/fskmeans/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from datasets import load_dataset
from itertools import islice
import numpy as np
import json
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

**DATASET WIKIPEDIA STRUCTURED**
*O conjunto de dados contém todos os artigos das edições em inglês e francês da Wikipédia, pré-analisados ​​e apresentados como dados estruturados com um esquema consistente. O conjunto de dados é fornecido no formato Parquet, otimizado para consultas analíticas de alto desempenho e armazenamento eficiente. Esta versão utiliza um esquema unificado e fixo em todos os arquivos, tornando-a compatível com DuckDB, pandas, Polars e Apache Spark. Usamos "apenas" 1 milhao de linhas para obter aproximadamente 10 mil documentos com seccoes que superam as 1500 palavras*


*Formato do dataset:*

{
  "name": "Josephine Baker",
  "identifier": 255083,
  "url": "https://en.wikipedia.org/wiki/Josephine_Baker",
  "description": "American-born French dancer...",
  "abstract": "Freda Josephine Baker...",
  "main_entity": {
    "identifier": "Q151972",
    "url": "https://www.wikidata.org/entity/Q151972"
  },
  "version": {
    "identifier": 123456789,
    "editor": {
      "identifier": 123,
      "name": "Example editor"
    },
    "scores": {}
  },
  "image": {
    "content_url": "https://upload.wikimedia.org/...",
    "width": 250,
    "height": 350
  },
  "infoboxes": "[{...}]",
  "sections": "[{...}]",
  "tables": "[{...}]",
  "references": [
    {
      "identifier": "...",
      "metadata": "{...}"
    }
  ]
}*

In [2]:
#BLOCO FUNCIONAL - DECLARACAO DE FUNCOES:

def extrair_texto(doc):
    #extrair das secoes e concatenar em um unico texto

    try:
#devido a natureza semiestruturada, precisamos tratar excecoes para evitar erros
        texto = []
        secoes_dict = {}
        secoes = json.loads(doc["sections"])

        for secao in secoes:

            nome_secao = secao.get("name", "SEM_NOME")

            paragrafos_secao = []

            for parte in secao.get("has_parts", []):

                valor = parte.get("value")

                if (
                    parte.get("type") == "paragraph"
                    and isinstance(valor, str)
                ):

                    paragrafos_secao.append(valor)
                    texto.append(valor)

            if paragrafos_secao:

                secoes_dict[nome_secao] = " ".join(
                    paragrafos_secao
                )

        return {
            "texto": " ".join(texto),
            "secoes": secoes_dict
        }

    except Exception:

        return None
    
    #funcao que captura o numero de palavras de cada documento
def tamanho_documento(doc):

    try:
        secoes = json.loads(doc["sections"])
        palavras = 0
        for secao in secoes:
            if "has_parts" not in secao:
                continue
            for parte in secao["has_parts"]:
                if parte.get("type") == "paragraph":
                    palavras += len(
                        parte["value"].split()
                    )
        return palavras
    except:
        return None
#funcao de limpeza de texto
def limpar_texto(texto):
    #MINUSCULAS
    texto = texto.lower()
    #NORMALIZACAO - SEPARA CAFÉ EM CAFE + ´ 
    texto = unicodedata.normalize("NFKD", texto)
    #USANDO ASCII COMO PADRAO  - REMOVE ACENTOS
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    #SUBSTITUINDO - REGEX >>> NAO ACEITA COISAS QUE NAO SAO LETRAS NUMEROS OU ESPACOS
    # NO LUGAR METE UM ESPACO
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    #SUBSTITUINDO ESPACOS MAIORES QUE 1! POR UM ESPAÇO
    texto = re.sub(r"\s+", " ", texto)
    #STRIP REMOVE ESPACOS NO COMECO E FIM
    return texto.strip()

In [ ]:
#carregando o dataset, ATENCAO NAO RODAR CASO FOR LER DO JSON, VAI DEMORAR MUITO
#PULAR ESTE BLOCO
ds = load_dataset(
    "wikimedia/structured-wikipedia",
    "enwiki_namespace_0",
    streaming=True
)
#1 MILHAO DE INSTANCIAS FORNECERAM 18 MIL DOCUMENTOS, RODEI ISSO DUAS VEZES PRA
#CHEGARMOS A 35 MIL DOCUMENTOS

amostra = list(islice(ds["train"],1000000))

#APENAS PRA CONFERENCIA DO PROCESSO INICIAL
#texto = extrair_texto(amostra[0])
#print("Palavras:", len(texto["texto"].split()))

tamanhos = []

amostra_completa = []

for doc in amostra:

    resultado = extrair_texto(doc)
    if resultado:
        texto = resultado["texto"]
        secoes = resultado["secoes"]
        n_palavras = len(texto.split())
        if n_palavras > 1500:

            tamanhos.append(n_palavras)
            documento = {
                "id": doc["identifier"],
                "titulo": doc["name"],
                "url": doc["url"],
                "texto": texto,
                "secoes": secoes,
                "n_palavras": n_palavras
            }

            amostra_completa.append(documento)

p25, p50, p75, p90, p95, p99 = np.percentile(
    tamanhos,
    [25, 50, 75, 90, 95, 99]
)

print(f"P25: {p25:.0f}")
print(f"Mediana: {p50:.0f}")
print(f"P75: {p75:.0f}")
print(f"P90: {p90:.0f}")
print(f"P95: {p95:.0f}")
print(f"P99: {p99:.0f}")
print(f"Média: {np.mean(tamanhos):.0f}")
print(f"Máximo: {np.max(tamanhos)}")
print(len(tamanhos))


In [ ]:
##PERSISTINDO OS DADOS EM UM JSON PARA EVITAR STREAMMING CONSTANTE
#NAO PRECISA RODAR CASO PEGUE O ARQUIVO JSON JA CRIADO APENAS A PARTE FINAL DESCOMENTADA
#with open(
    #"corpus_wikipedia_longos.json",
    #"w",
    #encoding="utf-8"
#) as f:

    #json.dump(
       # amostra_completa,
        #f,
        #ensure_ascii=False,
        #indent=2
    #)

#POSTERIORMENTE QUANDO FORMOS USAR O CONTEUDO TEXTUAL
with open(
    "corpus_wikipedia_longos.json",
    "r",
encoding="utf-8"
) as f:

    amostra_completa = json.load(f)

#AQUI VOLTAMOS A TER OS DADOS EM MEMORIA NA VARIAVEL amostra_completa

In [ ]:
#dimensionando o vocabulario, inferindo a dimensionalidade dos dados

# MEDINDO O VOCABULARIO PRA VER COMO ESTAMOS COM RELACAO A MAX FEATURES
# entra como parametro para o vetorizador (max.features)
vocabulario = set()

for doc in amostra_completa:
    vocabulario.update(doc["texto"].split())

print(f"Vocabulario antes do tratamento: {len(vocabulario)}")

#tratamento do dataset, ajuda na reducao de dimensionalidade

#TRATAMENTO DE TEXTO, CONFORME FUNCAO DE LIMPEZA (MMINUSCULO, ACENTOS E ESPECIAIS)


for k in amostra_completa:
    k["texto"] = limpar_texto(k["texto"])

#VERIFICANDO VOCABULARIO APOS TRATAMENTO

vocabulario = set()
for doc in amostra_completa:
    vocabulario.update(doc["texto"].split())

print(f"Vocabulario apos o tratamento: {len(vocabulario)}")

##como o JSON PERSISTIDO ja esta tratado voce vera valores iguais
#tinhamos 600k palavras reduzidas para 300


Vocabulario antes do tratamento: 308455
Vocabulario apos o tratamento: 308455


In [6]:
#vetorizaçao TF-IDF
textos = [doc["texto"] for doc in amostra_completa]
#CAPTURA LISTA DE TEXTOS

#DECLARACAO DO VETORIZADOR
vetorizador = TfidfVectorizer(
    stop_words="english",
    #max features da uma segurança contra alta dimensionalidade
    max_features=10000,
    min_df=2, #ignorar palavras que aparecem em menos de 3 documentos
    max_df=0.9, #ignorar palavras que aparecem em mais de 80% dos documentos
    #adicionamos os bigramas e trigramas para capturar um pouco de contexto
    #desejamos realizar testes com e sem esse parametro tao importante
    ngram_range=(1, 2) #bigramas e trigramas
)

#APLICAÇÃO DO VETORIZADOR
X = vetorizador.fit_transform(textos)
print(X.shape)  # (24, ≤3000)

#VERIFICANDO OS n-GRAMAS GERADOS
recursos = vetorizador.get_feature_names_out()

trigrams_gerados = [termo for termo in recursos if len(termo.split()) == 3]

print(f"Total de trigrams que entraram no vocabulário: {len(trigrams_gerados)}")
print("Exemplos de trigrams encontrados:")
print(trigrams_gerados[:30]) # Veja os 30 primeiros




(9362, 10000)
Total de trigrams que entraram no vocabulário: 0
Exemplos de trigrams encontrados:
[]


In [ ]:
#ESMAGAMENTO DE DADOS - reduzindo a dimensionalidade

svd = TruncatedSVD(n_components=100, random_state=42)
X_svd = svd.fit_transform(X)

print(X_svd.shape)  # (24, 100)

variancia_retida = np.sum(svd.explained_variance_ratio_) * 100
print(f"O SVD conseguiu reter {variancia_retida:.2f}% da informação original dos textos.")

termos = vetorizador.get_feature_names_out()

#visualizar os topicos latentes (3 PRIMEIROS), podemos ver mais tambem
for i, topico in enumerate(svd.components_[:3]):
    # Pega os índices dos 10 termos com maior peso no tópico
    top_termos_idx = topico.argsort()[::-1][:10]
    top_termos = [termos[idx] for idx in top_termos_idx]
    print(f"\nTópico Latente {i+1}:")
    print(", ".join(top_termos))

(9362, 100)
O SVD conseguiu reter 22.68% da informação original dos textos.

Tópico Latente 1:
album, new, school, time, film, war, music, song, church, city

Tópico Latente 2:
album, song, band, music, songs, released, tour, chart, video, rock

Tópico Latente 3:
municipality, swiss, population, sector, non swiss, 2000, apartments, school, total, album
